# Lab 4: Deep Learning with PyTorch - MNIST Classification

## 🎯 Learning Objectives

By the end of this lab, you will:
- Understand how PyTorch relates to micrograd
- Build a neural network using PyTorch's `nn.Module`
- Train on a real dataset (MNIST handwritten digits)
- Implement a complete training and validation pipeline
- Track and visualize training metrics

**What You'll Build:** A 2-layer MLP that classifies handwritten digits (0-9) with >95% accuracy

**Note:** PyTorch is pre-installed in Google Colab!

**Setup:** Check GPU availability

## Part 1: From Micrograd to PyTorch

### What's the Same?

PyTorch works **exactly like micrograd**, just faster and with more features:

| Concept | Micrograd | PyTorch |
|---------|-----------|----------|
| Tracking values | `Value` | `torch.Tensor` |
| Autograd | `backward()` | `backward()` |
| Neural network | `MLP` | `nn.Module` |
| Layer | `Layer` | `nn.Linear` |
| Activation | `.relu()` | `F.relu()` or `nn.ReLU()` |
| Parameters | `.parameters()` | `.parameters()` |
| Optimization | Manual loop | `torch.optim` |

### What's Different?

1. **Speed:** PyTorch uses GPU acceleration (1000x faster)
2. **Tensors:** Multi-dimensional arrays (not just scalars)
3. **Built-in layers:** Don't need to implement Neuron/Layer
4. **Optimizers:** SGD, Adam, etc. (not manual gradient descent)
5. **More operations:** Convolutions, attention, etc.

### The Core Idea Remains the Same:

```python
# Forward pass
output = model(input)
loss = compute_loss(output, target)

# Backward pass
loss.backward()  # ← Same as micrograd!

# Update weights
optimizer.step()
```

### 📚 Recap: Lab 3 Custom Autograd

In Lab 3, you implemented a custom autograd engine from scratch:
- Built `Value` class to track operations
- Implemented `_backward()` functions for each operation
- Used topological sort for backpropagation

**You understood HOW autograd works internally!**

Now in Lab 4, you'll use PyTorch's production-ready autograd:
- Same concepts, different API
- Much faster (optimized C++/CUDA code)
- Supports multi-dimensional tensors

**Learn More:** [PyTorch Autograd Tutorial](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html) - Official guide covering the same concepts you implemented in Lab 3

## Part 2: PyTorch Basics

### Tensors - PyTorch's "Value" Objects

A tensor is like a multi-dimensional array that tracks gradients:

**API Reference:** [torch.Tensor](https://pytorch.org/docs/stable/tensors.html)

In [ ]:
import torch

print("=" * 60)
print("Creating Tensors")
print("=" * 60)

# From Python list
x = torch.tensor([1.0, 2.0, 3.0])
print(f"1D tensor: {x}")
print(f"Shape: {x.shape}")
print(f"Type: {x.dtype}\n")

# 2D tensor (matrix)
y = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(f"2D tensor:\n{y}")
print(f"Shape: {y.shape}\n")

# Tensor with gradient tracking
z = torch.tensor([5.0], requires_grad=True)  # ← Like Value!
print(f"Tensor with gradient tracking: {z}")
print(f"Requires grad: {z.requires_grad}")

### PyTorch Autograd Demo

In [ ]:
# Just like micrograd!
a = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([3.0], requires_grad=True)
c = a + b
d = c * 2

print(f"Forward: d = (a + b) * 2 = {d.item()}")

# Backward pass
d.backward()

print(f"\nBackward:")
print(f"a.grad = {a.grad.item()}  (dd/da)")
print(f"b.grad = {b.grad.item()}  (dd/db)")
print(f"\n✓ Same as micrograd, but with tensors!")

## Part 3: The MNIST Dataset

### What is MNIST?

MNIST is the "Hello World" of deep learning:
- 70,000 images of handwritten digits (0-9)
- 28×28 grayscale images
- 60,000 training + 10,000 test images

**Dataset Reference:** [scikit-learn digits dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)

We'll use a smaller version (8×8 images) from sklearn for faster training:

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Load dataset
digits = load_digits()
X_data = digits.data  # Shape: (1797, 64) - flattened 8x8 images
y_data = digits.target  # Shape: (1797,) - labels 0-9

print(f"Dataset: {X_data.shape[0]} samples")
print(f"Features: {X_data.shape[1]} (8×8 = 64 pixels)")
print(f"Classes: 10 (digits 0-9)")

# Visualize some examples
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_data[i].reshape(8, 8), cmap='gray')
    ax.set_title(f"Label: {y_data[i]}")
    ax.axis('off')
plt.suptitle("Sample MNIST Digits")
plt.show()

### Prepare Train/Test Split

In [ ]:
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
X_train = torch.FloatTensor(X_train)
y_train = torch.LongTensor(y_train)
X_test = torch.FloatTensor(X_test)
y_test = torch.LongTensor(y_test)

# Normalize to [0, 1]
X_train = X_train / 16.0  # Pixel values are 0-16
X_test = X_test / 16.0

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## Exercise 1: Build MLP with PyTorch

### Your Task

Implement a 2-layer MLP using PyTorch:
- Input: 64 features (8×8 pixels)
- Hidden layer 1: 128 neurons with ReLU
- Hidden layer 2: 64 neurons with ReLU
- Output: 10 neurons (one per class, no activation)

**API Reference:**
- [torch.nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)
- [torch.nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
- [torch.nn.functional.relu](https://pytorch.org/docs/stable/generated/torch.nn.functional.relu.html)

### Starter Code

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    """2-layer MLP for MNIST classification."""
    
    def __init__(self):
        super().__init__()
        # self.fc1 = nn.Linear(64, 128)   # Input to hidden 1
        # self.fc2 = nn.Linear(128, 64)   # Hidden 1 to hidden 2
        # self.fc3 = nn.Linear(64, 10)    # Hidden 2 to output
        raise NotImplementedError("Define layers")
    
    def forward(self, x):
        """
        Forward pass through network.
        
        Args:
            x: Input tensor of shape (batch_size, 64)
        
        Returns:
            Output tensor of shape (batch_size, 10)
        
        1. Pass x through fc1, then apply ReLU
           x = F.relu(self.fc1(x))
        2. Pass through fc2, then apply ReLU
           x = F.relu(self.fc2(x))
        3. Pass through fc3 (no activation for output)
           x = self.fc3(x)
        4. Return x
        """
        raise NotImplementedError("Implement forward pass")

### Test Your Model

In [ ]:
# Create model
model = MLP()
print(f"Model architecture:\n{model}\n")

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params:,}")
# Expected: (64*128 + 128) + (128*64 + 64) + (64*10 + 10) = 8192 + 128 + 8192 + 64 + 640 + 10 = 17,226

# Test forward pass
sample_input = X_train[:5]  # First 5 samples
sample_output = model(sample_input)
print(f"\nInput shape: {sample_input.shape}")
print(f"Output shape: {sample_output.shape}  (should be [5, 10])")
assert sample_output.shape == (5, 10)
print("\n✓ Model works!")

## Exercise 2: Implement Training Loop

### Training Components

1. **Loss Function:** Cross-entropy loss for classification
   - API: [torch.nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)

2. **Optimizer:** Adam optimizer (better than plain gradient descent)
   - API: [torch.optim.Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html)

### The Training Loop Pattern

```python
for epoch in range(num_epochs):
    # Forward pass
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    
    # Backward pass
    optimizer.zero_grad()  # Reset gradients
    loss.backward()         # Compute gradients
    optimizer.step()        # Update weights
```

### Your Task

Implement the training loop:

In [ ]:
import torch.optim as optim
from tqdm import tqdm

# Create fresh model
model = MLP()

num_epochs = 10  # Reduced from 50 for faster training

# Lists to track metrics
train_losses = []
train_accs = []

print("\nUncomment the code above to train!")

### What You Should See

During training:
```
Epoch [10/50], Loss: 0.8234, Accuracy: 75.32%
Epoch [20/50], Loss: 0.3421, Accuracy: 89.67%
Epoch [30/50], Loss: 0.1532, Accuracy: 95.23%
Epoch [40/50], Loss: 0.0834, Accuracy: 97.45%
Epoch [50/50], Loss: 0.0523, Accuracy: 98.21%
```

- Loss should decrease
- Accuracy should increase
- Final training accuracy should be > 95%

## Exercise 3: Implement Validation Loop

### Why Validation?

Training accuracy can be misleading (model might memorize).
We need to test on **unseen data** to measure true performance.

### Key Differences from Training:

1. **No gradient computation:** Use `with torch.no_grad():`
2. **Eval mode:** Call `model.eval()`
3. **No optimizer.step():** Just measure performance

### Your Task

Implement validation to measure test accuracy:

In [ ]:
# 
# model.eval()  # Set to evaluation mode
# 
# with torch.no_grad():  # Don't compute gradients
#     # Forward pass on test data
#     test_outputs = model(X_test)
#     
#     # Get predictions
#     _, predicted = torch.max(test_outputs.data, 1)
#     
#     # Calculate accuracy
#     test_accuracy = (predicted == y_test).float().mean()
#     
#     # Calculate loss
#     test_loss = criterion(test_outputs, y_test)
# 
# print(f"\nTest Results:")
# print(f"Test Loss: {test_loss.item():.4f}")
# print(f"Test Accuracy: {test_accuracy.item()*100:.2f}%")

print("\nUncomment the code above to validate!")

### Expected Results

A well-trained model should achieve:
- **Test Accuracy:** > 95%
- **Test Loss:** < 0.2

If your accuracy is lower, try:
- Training for more epochs
- Adjusting learning rate
- Adding more neurons to hidden layers

## Part 4: Visualizing Training Progress

### Plot Training Curves

In [ ]:
# Uncomment after training:

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# # Plot loss
# ax1.plot(train_losses)
# ax1.set_xlabel('Epoch')
# ax1.set_ylabel('Loss')
# ax1.set_title('Training Loss')
# ax1.grid(True)

# # Plot accuracy
# ax2.plot([acc*100 for acc in train_accs], label='Train')
# ax2.set_xlabel('Epoch')
# ax2.set_ylabel('Accuracy (%)')
# ax2.set_title('Training Accuracy')
# ax2.legend()
# ax2.grid(True)

# plt.tight_layout()
# plt.show()

### Visualize Predictions

In [ ]:
# Uncomment after training:

# # Show some predictions
# model.eval()
# with torch.no_grad():
#     test_outputs = model(X_test[:10])
#     _, predicted = torch.max(test_outputs, 1)

# fig, axes = plt.subplots(2, 5, figsize=(10, 4))
# for i, ax in enumerate(axes.flat):
#     img = X_test[i].numpy().reshape(8, 8)
#     ax.imshow(img, cmap='gray')
#     correct = predicted[i] == y_test[i]
#     color = 'green' if correct else 'red'
#     ax.set_title(f"Pred: {predicted[i].item()}\nTrue: {y_test[i].item()}", color=color)
#     ax.axis('off')
# plt.suptitle("Test Predictions (Green=Correct, Red=Wrong)")
# plt.show()

## Bonus: Confusion Matrix

See which digits are confused with each other:

In [ ]:
# Uncomment after training:

# from sklearn.metrics import confusion_matrix
# import seaborn as sns

# model.eval()
# with torch.no_grad():
#     test_outputs = model(X_test)
#     _, predicted = torch.max(test_outputs, 1)

# cm = confusion_matrix(y_test.numpy(), predicted.numpy())

# plt.figure(figsize=(8, 6))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
# plt.xlabel('Predicted')
# plt.ylabel('True')
# plt.title('Confusion Matrix')
# plt.show()

# print("\nDiagonal = correct predictions")
# print("Off-diagonal = misclassifications")

## Complete Training Pipeline

Run everything together:

In [ ]:
# Uncomment to run complete pipeline:

# print("Starting training...\n")

# # 1. Create model
# model = MLP()
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001)

# # 2. Train
# for epoch in range(50):
#     model.train()
#     outputs = model(X_train)
#     loss = criterion(outputs, y_train)
#     
#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()
#     
#     if (epoch + 1) % 10 == 0:
#         _, predicted = torch.max(outputs.data, 1)
#         acc = (predicted == y_train).float().mean()
#         print(f"Epoch [{epoch+1}/50], Loss: {loss.item():.4f}, Acc: {acc.item()*100:.2f}%")

# # 3. Validate
# model.eval()
# with torch.no_grad():
#     test_outputs = model(X_test)
#     _, predicted = torch.max(test_outputs, 1)
#     test_acc = (predicted == y_test).float().mean()
#     print(f"\nFinal Test Accuracy: {test_acc.item()*100:.2f}%")

# print("\n🎉 Training complete!")

## Summary

### What You've Learned

✅ **PyTorch basics:** Tensors, autograd, nn.Module

✅ **Building networks:** Define architecture with nn.Linear and activations

✅ **Training loop:** Forward pass, backward pass, optimizer step

✅ **Validation:** Measure performance on test data

✅ **Real dataset:** MNIST handwritten digits

✅ **Complete pipeline:** Data loading → training → validation → visualization

### Micrograd vs PyTorch

**What's the Same:**
- Computation graphs track operations ✓
- `backward()` computes gradients ✓
- Neural networks are composed of layers ✓
- Training updates weights using gradients ✓

**What PyTorch Adds:**
- Multi-dimensional tensors (not just scalars)
- GPU acceleration (1000x faster)
- Built-in layers, optimizers, loss functions
- Data loaders for efficient batching
- Production-ready features

### You've Completed the Journey!

From Lab 1 to Lab 4:
1. **Lab 1:** Built arrays from scratch (understood NumPy)
2. **Lab 2:** Built computation graphs (understood autograd)
3. **Lab 3:** Built neural networks from scratch (understood architecture)
4. **Lab 4:** Used PyTorch on real data (understood production systems)

**You now understand deep learning from first principles!**

### Next Steps

- Try different architectures (deeper networks, different activations)
- Experiment with other datasets (CIFAR-10, Fashion-MNIST)
- Learn about CNNs for image classification
- Learn about RNNs/Transformers for sequences
- Build projects and share them!

**Congratulations!** 🎉 You've mastered the fundamentals of deep learning!